# Modelos base del proyecto

## Set up y carga

In [1]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_config
from src.preprocessing import load_split
from src.train import compute_class_weights
from src.evaluate import evaluate_and_log, save_predictions, pr_auc

cfg = load_config()
from src.runtime import limit_threads
N_THREADS = limit_threads(cfg)
SEED = cfg["seeds"][0]
print(cfg["baselines"]["xgboost"]["early_stopping_rounds"], cfg["nn"]["search"], cfg["results"]["path"])

tr, dv = load_split(cfg, "train"), load_split(cfg, "dev")
X_tr, y_tr = tr["X"], tr["y"]["isFraud"]
X_dv, y_dv = dv["X"], dv["y"]["isFraud"]

assert list(X_tr.columns) == list(X_dv.columns)
print(X_tr.shape, X_dv.shape, f"fraude train {y_tr.mean():.3%} | dev {y_dv.mean():.3%}")
print("NaN:", int(X_tr.isna().sum().sum()), int(X_dv.isna().sum().sum()))

# XGBoost no acepta [, ] ni < en los nombres de columna
bad = [c for c in X_tr.columns if any(ch in c for ch in "[]<")]
print("nombres problemáticos para XGBoost:", bad)

100 {'n_trials': 10, 'seed': 19} reports/results.csv
(413378, 383) (88581, 383) fraude train 3.517% | dev 3.434%
NaN: 0 0
nombres problemáticos para XGBoost: []


## Pesos de clase

In [2]:
cw = compute_class_weights(y_tr)
n0, n1 = int((y_tr == 0).sum()), int((y_tr == 1).sum())
spw = cfg["baselines"]["xgboost"]["scale_pos_weight"] or n0 / n1
print("class_weight:", cw, "| scale_pos_weight:", round(spw, 2))
# esperado: {0: ~0.518, 1: ~14.217} y 27.43

class_weight: {'class_weight': {0: 0.5182253535252231, 1: 14.217155041959003}, 'scale_pos_weight': 27.434310083918007, 'n0': 398840, 'n1': 14538} | scale_pos_weight: 27.43


# Clasificadores base


## Sin información 
Una puntuación constante da PR-AUC igual a la tasa de fraude y ROC-AUC de 0.5

In [3]:
m = evaluate_and_log(y_dv, np.full(len(y_dv), 0.5), cfg, stage="baseline", model="no_info",
                     variant="constante", seed=SEED, n_features=0,
                     notes="piso: PR-AUC = tasa de fraude en dev")
print(f"PR-AUC {m['pr_auc']:.4f} (piso dev 0.0343) | ROC-AUC {m['roc_auc']:.3f}")

PR-AUC 0.0343 (piso dev 0.0343) | ROC-AUC 0.500


## Regresión Logística 

Buscando el hiperparametro C

In [4]:
from sklearn.linear_model import LogisticRegression

lr_cfg = cfg["baselines"]["logreg"]
lr_res = {}
for C in lr_cfg["C_grid"]:
    t0 = time.perf_counter()
    lr = LogisticRegression(C=C, class_weight=cw["class_weight"], max_iter=lr_cfg["max_iter"], solver="lbfgs")
    lr.fit(X_tr, y_tr)
    t = time.perf_counter() - t0
    p_dv, p_tr = lr.predict_proba(X_dv)[:, 1], lr.predict_proba(X_tr)[:, 1]
    m = evaluate_and_log(y_dv, p_dv, cfg, stage="baseline", model="logreg", variant=f"C={C}",
                         cost_strategy="class_weight", seed=SEED,
                         hyperparams={"C": C, "max_iter": lr_cfg["max_iter"]},
                         n_features=X_tr.shape[1], best_iter=int(np.max(lr.n_iter_)),
                         train_time_s=round(t, 1), y_train=y_tr, p_train=p_tr)
    lr_res[C] = (m, p_dv)
    print(f"C={C:<5} PR-AUC dev {m['pr_auc']:.4f} | train {pr_auc(y_tr, p_tr):.4f} | "
          f"ROC {m['roc_auc']:.4f} | iter {int(np.max(lr.n_iter_))} | {t:.0f}s")

best_C = max(lr_res, key=lambda c: lr_res[c][0]["pr_auc"])
print("C elegido en dev:", best_C)
save_predictions("logreg_dev", lr_res[best_C][1], cfg)

C=0.01  PR-AUC dev 0.4498 | train 0.4913 | ROC 0.8620 | iter 193 | 9s
C=0.1   PR-AUC dev 0.4500 | train 0.4923 | ROC 0.8601 | iter 390 | 17s
C=1     PR-AUC dev 0.4491 | train 0.4929 | ROC 0.8581 | iter 472 | 22s
C=10    PR-AUC dev 0.4491 | train 0.4930 | ROC 0.8579 | iter 475 | 21s
C elegido en dev: 0.1


WindowsPath('C:/Users/kevin/fintech-fraud-detection-dl/reports/preds/logreg_dev.npy')

## XGBoost 

In [5]:
import xgboost as xgb

xg_cfg = cfg["baselines"]["xgboost"]
t0 = time.perf_counter()
xgb_model = xgb.XGBClassifier(**xg_cfg["params"], n_jobs=N_THREADS, scale_pos_weight=spw,
                              early_stopping_rounds=xg_cfg["early_stopping_rounds"],
                              random_state=SEED)
xgb_model.fit(X_tr, y_tr, eval_set=[(X_dv, y_dv)], verbose=100)
t = time.perf_counter() - t0

bi = int(xgb_model.best_iteration)
p_dv = xgb_model.predict_proba(X_dv, iteration_range=(0, bi + 1))[:, 1]
p_tr = xgb_model.predict_proba(X_tr, iteration_range=(0, bi + 1))[:, 1]
m = evaluate_and_log(y_dv, p_dv, cfg, stage="baseline", model="xgboost", variant="default",
                     cost_strategy="scale_pos_weight", seed=SEED,
                     hyperparams={**xg_cfg["params"], "scale_pos_weight": round(spw, 3)},
                     n_features=X_tr.shape[1], best_iter=bi, train_time_s=round(t, 1),
                     y_train=y_tr, p_train=p_tr,
                     notes="n_estimators elegido con early stopping en dev (PR-AUC dev algo optimista)")
save_predictions("xgboost_dev", p_dv, cfg)
print(f"best_iter {bi} | PR-AUC dev {m['pr_auc']:.4f} | train {pr_auc(y_tr, p_tr):.4f} | "
      f"ROC {m['roc_auc']:.4f} | recall@P0.5 {m['recall_at_p']:.3f} | {t:.0f}s")

[0]	validation_0-aucpr:0.31177
[100]	validation_0-aucpr:0.47708
[200]	validation_0-aucpr:0.50520
[300]	validation_0-aucpr:0.52244
[400]	validation_0-aucpr:0.53228
[500]	validation_0-aucpr:0.53782
[600]	validation_0-aucpr:0.54342
[700]	validation_0-aucpr:0.54888
[800]	validation_0-aucpr:0.55026
[900]	validation_0-aucpr:0.55182
[1000]	validation_0-aucpr:0.55233
[1100]	validation_0-aucpr:0.55399
[1200]	validation_0-aucpr:0.55462
[1300]	validation_0-aucpr:0.55656
[1400]	validation_0-aucpr:0.55776
[1500]	validation_0-aucpr:0.55630
[1503]	validation_0-aucpr:0.55625
best_iter 1403 | PR-AUC dev 0.5580 | train 0.9337 | ROC 0.9094 | recall@P0.5 0.529 | 116s


## Comparación de los baselines

In [6]:
res = pd.read_csv(ROOT / cfg["results"]["path"])
cols = ["exp_id", "model", "variant", "cost_strategy", "best_iter", "train_time_s",
        "pr_auc_train", "pr_auc", "roc_auc", "recall_at_p", "f1_max", "min_cost_rel"]
res[res.stage == "baseline"][cols].round(4)

,exp_id,model,variant,cost_strategy,best_iter,train_time_s,pr_auc_train,pr_auc,roc_auc,recall_at_p,f1_max,min_cost_rel
0,exp_0001,no_info,constante,NaN,NaN,NaN,NaN,0.0343,0.5000,0.0000,0.0664,1.0000
1,exp_0002,logreg,C=0.01,class_weight,193.0,9.2,0.4913,0.4498,0.8620,0.4129,0.4542,0.5620
2,exp_0003,logreg,C=0.1,class_weight,390.0,17.3,0.4923,0.4500,0.8601,0.4083,0.4529,0.5609
3,exp_0004,logreg,C=1,class_weight,472.0,22.1,0.4929,0.4491,0.8581,0.4103,0.4520,0.5622
4,exp_0005,logreg,C=10,class_weight,475.0,20.9,0.4930,0.4491,0.8579,0.4109,0.4520,0.5631
5,exp_0006,xgboost,default,scale_pos_weight,1403.0,115.9,0.9337,0.5580,0.9094,0.5293,0.5417,0.4842


# Ablaciones

## funciones de ajuste

In [7]:
from src.evaluate import paired_bootstrap_diff

def fit_logreg(cols, C):
    t0 = time.perf_counter()
    m = LogisticRegression(C=C, class_weight=cw["class_weight"], max_iter=lr_cfg["max_iter"], solver="lbfgs")
    m.fit(X_tr[cols], y_tr)
    return (m.predict_proba(X_dv[cols])[:, 1], m.predict_proba(X_tr[cols])[:, 1],
            int(np.max(m.n_iter_)), time.perf_counter() - t0)

def fit_xgb(cols, seed=SEED):
    t0 = time.perf_counter()
    m = xgb.XGBClassifier(**xg_cfg["params"], n_jobs=N_THREADS, scale_pos_weight=spw,
                          early_stopping_rounds=xg_cfg["early_stopping_rounds"], random_state=seed)
    m.fit(X_tr[cols], y_tr, eval_set=[(X_dv[cols], y_dv)], verbose=False)
    bi = int(m.best_iteration)
    return (m.predict_proba(X_dv[cols], iteration_range=(0, bi + 1))[:, 1],
            m.predict_proba(X_tr[cols], iteration_range=(0, bi + 1))[:, 1],
            bi, time.perf_counter() - t0)

## definición de variantes

In [8]:
amt = cfg["ablations"]["amount_cols"]                                   # amt_cents, amt_freq
ind = [c for c in X_tr.columns if c.startswith(cfg["ablations"]["null_indicator_prefix"])]
dc = [c for c in X_tr.columns if any(c.startswith(f"{b}__") for b in cfg["columns"]["drop_candidates"])]
assert all(c in X_tr.columns for c in amt), "falta amt_cents o amt_freq en X"
print(f"indicadores de nulo: {len(ind)} (esperado ~33) | columnas de drop_candidates: {len(dc)} (esperado ~14)")

variants = {
    "ref_completo":            [],
    "imp_a_solo_amt_log":      amt,                                      # quita amt_cents y amt_freq
    "imp_b_amt_log+cents":     [c for c in amt if c != "amt_cents"],     # quita solo amt_freq
    "nulos_c_sin_indicadores": ind,
    "drop_candidates":         dc,
}
notes = {
    "nulos_c_sin_indicadores": "quita null_pat_*; ninguna dummy 'missing' sobrevive en X, así que se pierde casi toda la señal explícita de ausencia",
    "drop_candidates": f"quita {len(dc)} columnas de card4, M5, id_23, id_27, id_34",
}

indicadores de nulo: 33 (esperado ~33) | columnas de drop_candidates: 14 (esperado ~14)


## Correr ablaciones


In [9]:
preds = {}
for name, drop in variants.items():
    cols = [c for c in X_tr.columns if c not in set(drop)]
    p_dv, p_tr, it, t = fit_logreg(cols, best_C)
    evaluate_and_log(y_dv, p_dv, cfg, stage="ablation", model="logreg", variant=name,
                     cost_strategy="class_weight", seed=SEED, hyperparams={"C": best_C},
                     n_features=len(cols), best_iter=it, train_time_s=round(t, 1),
                     y_train=y_tr, p_train=p_tr, notes=notes.get(name, ""))
    preds[("logreg", name)] = p_dv

    p_dv, p_tr, bi, t = fit_xgb(cols)
    m = evaluate_and_log(y_dv, p_dv, cfg, stage="ablation", model="xgboost", variant=name,
                         cost_strategy="scale_pos_weight", seed=SEED,
                         hyperparams={**xg_cfg["params"], "scale_pos_weight": round(spw, 3)},
                         n_features=len(cols), best_iter=bi, train_time_s=round(t, 1),
                         y_train=y_tr, p_train=p_tr, notes=notes.get(name, ""))
    preds[("xgboost", name)] = p_dv
    print(f"{name:<26} {len(cols)} cols | XGB PR-AUC dev {m['pr_auc']:.4f} | {t:.0f}s")

ref_completo               383 cols | XGB PR-AUC dev 0.5580 | 125s
imp_a_solo_amt_log         381 cols | XGB PR-AUC dev 0.5589 | 111s
imp_b_amt_log+cents        382 cols | XGB PR-AUC dev 0.5655 | 143s
nulos_c_sin_indicadores    350 cols | XGB PR-AUC dev 0.5709 | 157s
drop_candidates            369 cols | XGB PR-AUC dev 0.5609 | 114s


## Comparación contra la referencia, con intervalo.

In [10]:
rows = []
for model in ["logreg", "xgboost"]:
    p_ref = preds[(model, "ref_completo")]
    for name in variants:
        if name == "ref_completo":
            continue
        d, lo, hi = paired_bootstrap_diff(y_dv, p_ref, preds[(model, name)], n=cfg["evaluation"]["bootstrap"]["n"], seed=SEED)
        rows.append({"modelo": model, "variante": name, "PR-AUC": pr_auc(y_dv, preds[(model, name)]),
                     "ref": pr_auc(y_dv, p_ref), "Δ vs ref": d, "IC95 inf": lo, "IC95 sup": hi,
                     "IC excluye 0": not (lo <= 0 <= hi)})
pd.DataFrame(rows).round(4)

,modelo,variante,PR-AUC,ref,Δ vs ref,IC95 inf,IC95 sup,IC excluye 0
0,logreg,imp_a_solo_amt_log,0.4494,0.450,-0.0007,-0.0015,0.0000,False
1,logreg,imp_b_amt_log+cents,0.4492,0.450,-0.0008,-0.0015,-0.0002,True
2,logreg,nulos_c_sin_indicadores,0.4415,0.450,-0.0084,-0.0112,-0.0050,True
3,logreg,drop_candidates,0.4478,0.450,-0.0022,-0.0037,-0.0009,True
4,xgboost,imp_a_solo_amt_log,0.5589,0.558,0.0006,-0.0042,0.0050,False
5,xgboost,imp_b_amt_log+cents,0.5655,0.558,0.0077,0.0040,0.0113,True
6,xgboost,nulos_c_sin_indicadores,0.5709,0.558,0.0131,0.0089,0.0176,True
7,xgboost,drop_candidates,0.5609,0.558,0.0032,-0.0007,0.0072,False


## Semillas en XGB


In [11]:
chk = {"ref_completo": [], "imp_b_amt_log+cents": variants["imp_b_amt_log+cents"],
       "nulos_c_sin_indicadores": ind, "drop_candidates": dc}
rows = []
for name, drop in chk.items():
    cols = [c for c in X_tr.columns if c not in set(drop)]
    for s in cfg["seeds"]:
        if s == SEED and ("xgboost", name) in preds:        # la semilla 19 ya se corrió
            rows.append({"variante": name, "seed": s, "pr_auc": pr_auc(y_dv, preds[("xgboost", name)])})
            continue
        p_dv, p_tr, bi, t = fit_xgb(cols, s)
        m = evaluate_and_log(y_dv, p_dv, cfg, stage="ablation", model="xgboost", variant=name,
                             cost_strategy="scale_pos_weight", seed=s,
                             hyperparams={**xg_cfg["params"], "scale_pos_weight": round(spw, 3)},
                             n_features=len(cols), best_iter=bi, train_time_s=round(t, 1),
                             y_train=y_tr, p_train=p_tr, notes="réplica de semilla de la ablación")
        rows.append({"variante": name, "seed": s, "pr_auc": m["pr_auc"]})
        print(f"{name:<26} semilla {s} | PR-AUC {m['pr_auc']:.4f} | {t:.0f}s")

piv = pd.DataFrame(rows).pivot(index="variante", columns="seed", values="pr_auc")
piv["media"] = piv[cfg["seeds"]].mean(axis=1)
piv["sd"] = piv[cfg["seeds"]].std(axis=1)
piv["Δ media vs ref"] = piv["media"] - piv.loc["ref_completo", "media"]
piv.round(4)

ref_completo               semilla 69 | PR-AUC 0.5727 | 127s
ref_completo               semilla 79 | PR-AUC 0.5643 | 146s
imp_b_amt_log+cents        semilla 69 | PR-AUC 0.5618 | 122s
imp_b_amt_log+cents        semilla 79 | PR-AUC 0.5688 | 159s
nulos_c_sin_indicadores    semilla 69 | PR-AUC 0.5663 | 111s
nulos_c_sin_indicadores    semilla 79 | PR-AUC 0.5606 | 76s
drop_candidates            semilla 69 | PR-AUC 0.5598 | 124s
drop_candidates            semilla 79 | PR-AUC 0.5536 | 86s


seed,19,69,79,media,sd,Δ media vs ref
variante,,,,,,
drop_candidates,0.5609,0.5598,0.5536,0.5581,0.0040,-0.0069
imp_b_amt_log+cents,0.5655,0.5618,0.5688,0.5654,0.0035,0.0004
nulos_c_sin_indicadores,0.5709,0.5663,0.5606,0.5659,0.0051,0.0009
ref_completo,0.5580,0.5727,0.5643,0.5650,0.0074,0.0000
